# Start Partition Comparison

This notebook compares the available start partition algorithms for the Dense Graph Partition experiments.

The comparison is performed separately for:

- Powerlaw and Erdős–Rényi graphs,
- sparse and dense instances,
- small and large instances.

Only two aggregated metrics are reported:

- **mean relative to best**: mean quotient between the solution density of an algorithm and the best density found on the same instance;
- **mean runtime**: mean runtime in seconds.

A value close to `1.0` for the relative solution quality indicates that an algorithm produces solutions close to the best available result.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
RESULTS_FILE = Path("../../results/experiment1/raw_results.csv")

ALGORITHM_ORDER = [
    "singleton",
    "matching",
    "maximum_matching",
    "maximum_matching_edge_cover",
    "high_degree_first_matching",
    "high_degree_product_matching",
    "kapoce",
    "leiden_mdgp",
]

GRAPH_ORDER = ["powerlaw", "er"]
REGIME_ORDER = ["sparse", "dense"]
SIZE_ORDER = ["small", "large"]

## Load experiment results

The raw experiment results are loaded and checked for the columns required by this analysis.

In [4]:
raw = pd.read_csv(RESULTS_FILE)

required_columns = {
    "graph_type",
    "regime",
    "size_class",
    "dataset",
    "instance",
    "algorithm",
    "relative_to_best",
    "runtime",
}

missing_columns = required_columns.difference(raw.columns)

if missing_columns:
    raise ValueError( "The result file is missing required columns: " + ", ".join(sorted(missing_columns)))

print(f"Loaded {len(raw):,} result rows.")

Loaded 16,000 result rows.


## Aggregate solution quality and runtime

For every combination of graph type, density regime, size class, and start partition algorithm, the notebook calculates:

1. the mean relative solution quality;
2. the mean runtime in seconds.

Each instance contributes one observation to the corresponding group.

In [5]:
summary = (
    raw
    .groupby(
        ["graph_type", "regime", "size_class", "algorithm"],
        as_index=False,
        observed=True,
    )
    .agg(
        mean_relative_to_best=("relative_to_best", "mean"),
        mean_runtime_seconds=("runtime", "mean"),
    )
)

summary["graph_type"] = pd.Categorical(
    summary["graph_type"],
    categories=GRAPH_ORDER,
    ordered=True,
)
summary["regime"] = pd.Categorical(
    summary["regime"],
    categories=REGIME_ORDER,
    ordered=True,
)
summary["size_class"] = pd.Categorical(
    summary["size_class"],
    categories=SIZE_ORDER,
    ordered=True,
)

present_algorithms = summary["algorithm"].unique().tolist()
algorithm_order = [
    algorithm
    for algorithm in ALGORITHM_ORDER
    if algorithm in present_algorithms
]
algorithm_order += sorted(set(present_algorithms).difference(algorithm_order))

summary["algorithm"] = pd.Categorical(
    summary["algorithm"],
    categories=algorithm_order,
    ordered=True,
)

summary = (
    summary
    .sort_values(["graph_type", "size_class", "regime", "algorithm"])
    .reset_index(drop=True)
)

summary

,graph_type,regime,size_class,algorithm,mean_relative_to_best,mean_runtime_seconds
0,powerlaw,sparse,small,singleton,inf,0.000030
1,powerlaw,sparse,small,matching,1.410530,0.000336
2,powerlaw,sparse,small,maximum_matching,1.114775,0.019037
3,powerlaw,sparse,small,maximum_matching_edge_cover,1.111345,0.017708
4,powerlaw,sparse,small,high_degree_first_matching,1.489544,0.001751
...,...,...,...,...,...,...
59,er,dense,large,maximum_matching_edge_cover,1.330450,1.148428
60,er,dense,large,high_degree_first_matching,1.375946,0.064794
61,er,dense,large,high_degree_product_matching,1.375946,0.033409
62,er,dense,large,kapoce,1.000000,0.229682


In [6]:
final_table = summary[
    [
        "graph_type",
        "size_class",
        "regime",
        "algorithm",
        "mean_relative_to_best",
        "mean_runtime_seconds",
    ]
].copy()

final_table["mean_relative_to_best"] = final_table["mean_relative_to_best"].round(4)
final_table["mean_runtime_seconds"] = final_table["mean_runtime_seconds"].round(5)

final_table

,graph_type,size_class,regime,algorithm,mean_relative_to_best,mean_runtime_seconds
0,powerlaw,small,sparse,singleton,inf,0.00003
1,powerlaw,small,sparse,matching,1.4105,0.00034
2,powerlaw,small,sparse,maximum_matching,1.1148,0.01904
3,powerlaw,small,sparse,maximum_matching_edge_cover,1.1113,0.01771
4,powerlaw,small,sparse,high_degree_first_matching,1.4895,0.00175
...,...,...,...,...,...,...
59,er,large,dense,maximum_matching_edge_cover,1.3304,1.14843
60,er,large,dense,high_degree_first_matching,1.3759,0.06479
61,er,large,dense,high_degree_product_matching,1.3759,0.03341
62,er,large,dense,kapoce,1.0000,0.22968


## Empirical approximation bound for KaPoCE

The maximum-cardinality matching construction is a 2-approximation for MDGP. Its objective value therefore provides an upper bound on the unknown optimum.

For every instance, the quality of the KaPoCE solution can consequently be bounded relative to the optimum using the ratio between KaPoCE and maximum matching.

In [8]:
instance_keys = [
    "graph_type",
    "regime",
    "size_class",
    "dataset",
    "instance",
]

maximum_matching = (
    raw[raw["algorithm"] == "maximum_matching"]
    [instance_keys + ["relative_to_best"]]
    .rename(columns={"relative_to_best": "maximum_matching_relative_to_best"})
)

kapoce = (
    raw[raw["algorithm"] == "kapoce"]
    [instance_keys + ["relative_to_best"]]
    .rename(columns={"relative_to_best": "kapoce_relative_to_best"})
)

approximation_per_instance = maximum_matching.merge(
    kapoce,
    on=instance_keys,
    how="inner",
    validate="one_to_one",
)

approximation_per_instance["kapoce_approximation_bound"] = 2.0  * approximation_per_instance["kapoce_relative_to_best"]  / approximation_per_instance["maximum_matching_relative_to_best"]

In [10]:
approximation_summary = (
    approximation_per_instance
    .groupby(
        ["graph_type", "size_class", "regime"],
        as_index=False,
        observed=True,
    )
    .agg(max_approximation_bound=("kapoce_approximation_bound", "max"))
)

approximation_summary["graph_type"] = pd.Categorical(
    approximation_summary["graph_type"],
    categories=GRAPH_ORDER,
    ordered=True,
)

approximation_summary["size_class"] = pd.Categorical(
    approximation_summary["size_class"],
    categories=SIZE_ORDER,
    ordered=True,
)

approximation_summary["regime"] = pd.Categorical(
    approximation_summary["regime"],
    categories=REGIME_ORDER,
    ordered=True,
)

approximation_summary = (
    approximation_summary
    .sort_values(["graph_type", "size_class", "regime"])
    .reset_index(drop=True)
)

approximation_summary

,graph_type,size_class,regime,max_approximation_bound
0,powerlaw,small,sparse,1.921599
1,powerlaw,small,dense,1.692047
2,powerlaw,large,sparse,1.898254
3,powerlaw,large,dense,1.646497
4,er,small,sparse,1.811102
5,er,small,dense,1.594261
6,er,large,sparse,1.949318
7,er,large,dense,1.576914


## LaTeX helper functions

The following functions format algorithm names and numerical values for the thesis table. Values are truncated rather than rounded, matching the formatting used in the move-operator notebook.

In [15]:
def truncate_number(value: float, decimals: int) -> float:
    factor = 10 ** decimals
    return np.trunc(value * factor) / factor


def latex_algorithm(algorithm: str) -> str:
    return r"\texttt{" + algorithm.replace("_", r"\_") + "}"


def format_number(value: float, decimals: int) -> str:
    return f"{truncate_number(value, decimals):.{decimals}f}"

## Build LaTeX comparison table

The table is grouped by graph type and dataset configuration. Within each dataset group:

- the highest mean relative solution quality is printed in bold

Ties are highlighted for all affected algorithms.

In [16]:
def make_start_partition_latex_table(df: pd.DataFrame, graph_type: str, caption: str, label: str) -> str:
    dataset_order = [
        ("small", "sparse"),
        ("small", "dense"),
        ("large", "sparse"),
        ("large", "dense"),
    ]

    graph_df = df[df["graph_type"] == graph_type].copy()

    if graph_df.empty:
        raise ValueError(f"No results available for graph type '{graph_type}'.")

    lines = [
        r"\begin{table}[t]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\begin{tabular}{p{2cm}p{6.3cm}p{3.1cm}p{2.1cm}}",
        r"\toprule",
        r"Datensatz & Startpartition & Mittlere relative Lösungsqualität & Mittlere Laufzeit (s) \\",
        r"\midrule",
    ]

    nonempty_datasets = [
        (size_class, regime)
        for size_class, regime in dataset_order
        if not graph_df[(graph_df["size_class"] == size_class) & (graph_df["regime"] == regime)].empty
    ]

    for dataset_index, (size_class, regime) in enumerate(nonempty_datasets):
        part = graph_df[(graph_df["size_class"] == size_class) & (graph_df["regime"] == regime)].copy()

        part["algorithm"] = pd.Categorical(part["algorithm"], categories=algorithm_order, ordered=True)
        part = part.sort_values("algorithm")

        best_quality = part["mean_relative_to_best"].min()

        dataset_label = f"{size_class} {regime}"

        for row_index, row in enumerate(part.itertuples(index=False)):
            dataset_cell = (
                rf"\multirow{{{len(part)}}}{{*}}{{{dataset_label}}}"
                if row_index == 0
                else ""
            )

            quality = format_number(row.mean_relative_to_best, 4)
            runtime = format_number(row.mean_runtime_seconds, 5)

            if np.isclose(row.mean_relative_to_best, best_quality):
                quality = rf"\textbf{{{quality}}}"

            lines.append(f"{dataset_cell} & {latex_algorithm(str(row.algorithm))} & {quality} & {runtime} \\\\")

        if dataset_index < len(nonempty_datasets) - 1:
            lines.append(r"\cmidrule(l){1-4}")

    lines.extend(
        [
            r"\bottomrule",
            r"\end{tabular}",
            r"\end{table}",
        ]
    )

    return "\n".join(lines)

In [17]:
powerlaw_latex = make_start_partition_latex_table(
    final_table,
    graph_type="powerlaw",
    caption=(
        "Mittlere relative Lösungsqualität und mittlere Laufzeit der Startpartitionsverfahren auf Powerlaw-Instanzen. Die relative Lösungsqualität ist der Quotient aus der besten auf derselben Instanz gefundenen Lösung und der Lösung des jeweiligen Verfahrens. Ein Wert von 1 entspricht der besten gefundenen Lösung."
    ),
    label="tab:start_partition_powerlaw",
)

print(powerlaw_latex)

\begin{table}[t]
\centering
\caption{Mittlere relative Lösungsqualität und mittlere Laufzeit der Startpartitionsverfahren auf Powerlaw-Instanzen. Die relative Lösungsqualität ist der Quotient aus der besten auf derselben Instanz gefundenen Lösung und der Lösung des jeweiligen Verfahrens. Ein Wert von 1 entspricht der besten gefundenen Lösung.}
\label{tab:start_partition_powerlaw}
\begin{tabular}{p{2cm}p{6.3cm}p{3.1cm}p{2.1cm}}
\toprule
Datensatz & Startpartition & Mittlere relative Lösungsqualität & Mittlere Laufzeit (s) \\
\midrule
\multirow{8}{*}{small sparse} & \texttt{singleton} & inf & 0.00003 \\
 & \texttt{matching} & 1.4105 & 0.00034 \\
 & \texttt{maximum\_matching} & 1.1148 & 0.01904 \\
 & \texttt{maximum\_matching\_edge\_cover} & 1.1113 & 0.01771 \\
 & \texttt{high\_degree\_first\_matching} & 1.4895 & 0.00175 \\
 & \texttt{high\_degree\_product\_matching} & 1.4895 & 0.00084 \\
 & \texttt{kapoce} & \textbf{1.0008} & 0.00989 \\
 & \texttt{leiden\_mdgp} & 1.0445 & 0.00120 \\
\cmi

In [18]:
er_latex = make_start_partition_latex_table(
    final_table,
    graph_type="er",caption=(
        "Mittlere relative Lösungsqualität und mittlere Laufzeit der Startpartitionsverfahren auf Erdős-Rényi-Instanzen. Die relative Lösungsqualität ist der Quotient aus der besten auf derselben Instanz gefundenen Lösung und der Lösung des jeweiligen Verfahrens. Ein Wert von 1 entspricht der besten gefundenen Lösung."
    ),
    label="tab:start_partition_er",
)

print(er_latex)

\begin{table}[t]
\centering
\caption{Mittlere relative Lösungsqualität und mittlere Laufzeit der Startpartitionsverfahren auf Erdős-Rényi-Instanzen. Die relative Lösungsqualität ist der Quotient aus der besten auf derselben Instanz gefundenen Lösung und der Lösung des jeweiligen Verfahrens. Ein Wert von 1 entspricht der besten gefundenen Lösung.}
\label{tab:start_partition_er}
\begin{tabular}{p{2cm}p{6.3cm}p{3.1cm}p{2.1cm}}
\toprule
Datensatz & Startpartition & Mittlere relative Lösungsqualität & Mittlere Laufzeit (s) \\
\midrule
\multirow{8}{*}{small sparse} & \texttt{singleton} & inf & 0.00003 \\
 & \texttt{matching} & 1.2709 & 0.00037 \\
 & \texttt{maximum\_matching} & 1.1661 & 0.01961 \\
 & \texttt{maximum\_matching\_edge\_cover} & 1.1627 & 0.01880 \\
 & \texttt{high\_degree\_first\_matching} & 1.3646 & 0.00316 \\
 & \texttt{high\_degree\_product\_matching} & 1.3646 & 0.00092 \\
 & \texttt{kapoce} & \textbf{1.0000} & 0.01235 \\
 & \texttt{leiden\_mdgp} & 1.1168 & 0.00120 \\
\cmidru

In [12]:
def make_approximation_latex_table(summary: pd.DataFrame) -> str:
    graph_order = ["powerlaw", "er"]
    dataset_order = [
        ("small", "sparse"),
        ("small", "dense"),
        ("large", "sparse"),
        ("large", "dense"),
    ]

    graph_labels = {
        "powerlaw": "Powerlaw",
        "er": "Erdős-Rényi",
    }

    lines = [
        r"\begin{table}[t]",
        r"\centering",
        r"\caption{Aus der 2-Approximationsgarantie des Maximum Matchings abgeleitete empirische Approximationsschranken für die von KaPoCE gefundenen Lösungen.}",
        r"\label{tab:kapoce_approximation}",
        r"\begin{tabular}{llr}",
        r"\toprule",
        r"Graphentyp & Datensatz & Approximationsschranke \\",
        r"\midrule",
    ]

    available_graphs = [
        graph_type
        for graph_type in graph_order
        if not summary[summary["graph_type"] == graph_type].empty
    ]

    for graph_index, graph_type in enumerate(available_graphs):
        graph_df = summary[summary["graph_type"] == graph_type].copy()

        rows = []

        for size_class, regime in dataset_order:
            part = graph_df[(graph_df["size_class"] == size_class) & (graph_df["regime"] == regime)]

            if not part.empty:
                rows.append(part.iloc[0])

        for row_index, row in enumerate(rows):
            graph_cell = (
                rf"\multirow{{{len(rows)}}}{{*}}{{{graph_labels[graph_type]}}}"
                if row_index == 0
                else ""
            )

            dataset_label = f"{row['size_class']} {row['regime']}"

            lines.append(f"{graph_cell} & {dataset_label} & {row['max_approximation_bound']:.3f} \\\\")

        if graph_index < len(available_graphs) - 1:
            lines.append(r"\midrule")

    lines.extend(
        [
            r"\bottomrule",
            r"\end{tabular}",
            r"\end{table}",
        ]
    )

    return "\n".join(lines)

In [13]:
approximation_table = make_approximation_latex_table(approximation_summary)
print(approximation_table)

\begin{table}[t]
\centering
\caption{Aus der 2-Approximationsgarantie des Maximum Matchings abgeleitete Approximationsschranken für die von KaPoCE gefundenen Lösungen.}
\label{tab:kapoce_approximation}
\begin{tabular}{llr}
\toprule
Graphentyp & Datensatz & Approximationsschranke \\
\midrule
\multirow{4}{*}{Powerlaw} & small sparse & 1.922 \\
 & small dense & 1.692 \\
 & large sparse & 1.898 \\
 & large dense & 1.646 \\
\midrule
\multirow{4}{*}{Erdős-Rényi} & small sparse & 1.811 \\
 & small dense & 1.594 \\
 & large sparse & 1.949 \\
 & large dense & 1.577 \\
\bottomrule
\end{tabular}
\end{table}
